# Clean Charges Pipeline
Go through the `Charges` column, clean it, extract statutes, split multiple charges into separate columns, and save as checkpoint 13.

**Steps:**
1. Load data and filter to rows with charges
2. Remove known junk entries (e.g. "LAWRENCE, MA")
3. Detect and flag noise / suspicious entries
4. Standardize text (lowercase, strip whitespace, etc.)
5. Extract statute references into a separate column
6. Split multiple charges into `charge_1`, `charge_2`, ... columns
7. Save cleaned checkpoint and a unique charges reference CSV

### Imports

In [24]:
import os
import re
import numpy as np
import pandas as pd

### Set up file paths
Define where the input checkpoint lives and where outputs will be saved.

In [25]:
import os

NOTEBOOK_DIR = os.getcwd()

DATA_DIR = os.path.join(NOTEBOOK_DIR, "..", "0_RecompiledScripts", "data", "missing_dates_csv")
CHECKPOINTS_DIR = os.path.join(DATA_DIR, "md_checkpoints")
CHARGES_DIR = os.path.join(DATA_DIR, "charges")

DATA_PATH = os.path.join(CHECKPOINTS_DIR, "checkpoint9_age.csv")
OUT_PATH = os.path.join(CHECKPOINTS_DIR, "checkpoint10_cleaned_charges.csv")
UNIQUE_CHARGES_PATH = os.path.join(CHARGES_DIR, "unique_charges_reference.csv")

os.makedirs(CHARGES_DIR, exist_ok=True)

print("Input:", os.path.abspath(DATA_PATH))
print("Output:", os.path.abspath(OUT_PATH))
print("Unique charges ref:", os.path.abspath(UNIQUE_CHARGES_PATH))

Input: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/data/missing_dates_csv/md_checkpoints/checkpoint9_age.csv
Output: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/data/missing_dates_csv/md_checkpoints/checkpoint10_cleaned_charges.csv
Unique charges ref: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/data/missing_dates_csv/charges/unique_charges_reference.csv


### Load checkpoint 9 and preview
Load the CSV and take a first look at the shape and the `Charges` column.

In [26]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nCharges non-null count:", df["Charges"].notna().sum())
print("Charges null count:", df["Charges"].isna().sum())
df.head()

Shape: (66741, 20)

Columns: ['Incident #', 'Date', 'Type', 'Location', 'Arrested', 'Location Prefix', 'DOB', 'Charges', 'raw_address', 'cleaned_address', 'latitude', 'longitude', 'geocode_confidence', 'geocoded_at', 'coord_valid', 'person_id', 'category', 'Year', 'crime_severity', 'Age']

Charges non-null count: 917
Charges null count: 65824


,Incident #,Date,Type,Location,Arrested,Location Prefix,DOB,Charges,raw_address,cleaned_address,latitude,longitude,geocode_confidence,geocoded_at,coord_valid,person_id,category,Year,crime_severity,Age
0,25026764.0,2025-07-01 00:12:08,THREATS,259 ANDOVER ST,No,NaN,NaN,NaN,259 ANDOVER ST,"259 ANDOVER ST, Lawrence, MA",42.694842,-71.155948,10.0,2026-06-22T23:33:38.335777+00:00,True,NaN,Violent and Weapon Offenses,2025,Serious,NaN
1,25026765.0,2025-07-01 00:19:36,TOW/REPOSSED,454 HAMPSHIRE ST,No,NaN,NaN,NaN,454 HAMPSHIRE ST,"454 HAMPSHIRE ST, Lawrence, MA",42.717278,-71.171768,10.0,2026-06-25T23:39:24.046150+00:00,True,NaN,Motor Vehicle Incidents,2025,Non-Serious,NaN
2,25026766.0,2025-07-01 00:49:07,DISTURBANCE,155 HAVERHILL ST,No,NaN,NaN,NaN,155 HAVERHILL ST,"155 HAVERHILL ST, Lawrence, MA",42.710380,-71.161736,10.0,2026-06-19T16:04:18.901970+00:00,True,NaN,Public Disturbances,2025,Non-Serious,NaN
3,25026767.0,2025-07-01 01:15:01,CK WELL BEING,621 COMMON ST,No,NaN,NaN,NaN,621 COMMON ST,"621 COMMON ST, Lawrence, MA",42.705034,-71.169622,10.0,2026-06-26T19:04:44.576031+00:00,True,NaN,Medical and Welfare Assistance,2025,Non-Serious,NaN
4,25026769.0,2025-07-01 01:29:06,EXTRA SURVEIL,425 MERRIMACK ST,No,NaN,NaN,NaN,425 MERRIMACK ST,"425 MERRIMACK ST, Lawrence, MA",42.703240,-71.143607,10.0,2026-06-25T22:01:03.147567+00:00,True,NaN,Preventive Policing,2025,Non-Serious,NaN


### Filter to rows that have non-empty charges
Keep only rows where `Charges` is not null and not blank. We keep a copy of the full dataframe so we can merge back later.

In [27]:
df_full = df.copy()

mask = df["Charges"].notna() & (df["Charges"].astype(str).str.strip() != "")
df = df[mask].copy().reset_index(drop=True)

print("Rows with charges:", df.shape[0])
print("Rows without charges (kept in df_full):", df_full.shape[0] - df.shape[0])
df["Charges"].head(10)

Rows with charges: 917
Rows without charges (kept in df_full): 65824


0                         LARCENY FROM PERSON c266 825
1    DEFAULT WARRANT: LARCENY FROM BUILDING C266 S8...
2    DEFAULT WARRANT: DRUG, POSSESS CLASS A c94C $3...
3    TRESPASS c266 S120; Warrant Charges: DEFAULT W...
4      DEFACEMENT MALICIOUS WANTON PROPERTY C266 S126A
5                      STANDARD WARRANT: MURDER 265 S1
6    DEFAULT WARRANT: RECKLESS OPERATION OF MOTOR; ...
7    LICENSE SUSPENDED, OP MV WITH c90 823; MARKED ...
8    DEFAULT WARRANT: UNLICENSED OPERATION OF MV c9...
9    DISORDERLY CONDUCT c272 S53; RESISTING ARREST ...
Name: Charges, dtype: object

### Remove known junk: "LAWRENCE, MA" variants
Some entries in the Charges column are just the city/state "LAWRENCE, MA" with various spacing. These are not real charges — set them to NaN.

In [28]:
# Match "LAWRENCE, MA", "LAWRENCE MA", "LAWRENCE,MA", "LAWRENCE , MA" etc.
lawrence_pattern = r"^\s*LAWRENCE\s*,?\s*MA\s*$"

lawrence_mask = df["Charges"].astype(str).str.contains(lawrence_pattern, case=False, na=False)
print(f"Entries matching 'LAWRENCE, MA' pattern: {lawrence_mask.sum()}")
print("\nSample of matched entries:")
print(df.loc[lawrence_mask, "Charges"].value_counts())

df.loc[lawrence_mask, "Charges"] = np.nan

Entries matching 'LAWRENCE, MA' pattern: 99

Sample of matched entries:
Charges
LAWRENCE, MA    99
Name: count, dtype: int64


### Detect suspicious / noisy entries
Look for entries that probably aren't real charges:
- Very short entries (under 3 characters)
- Purely numeric entries

We also scan for inline junk (URLs, dates, "PublicLog" artifacts) embedded in otherwise valid charges — these will be stripped out rather than deleting the whole entry.

In [29]:
charges_remaining = df["Charges"].dropna().astype(str)

# Very short entries (under 3 characters after stripping)
short_mask = charges_remaining.str.strip().str.len() < 3
short_entries = charges_remaining[short_mask]
print(f"=== Short entries (< 3 chars): {len(short_entries)} ===")
if len(short_entries) > 0:
    print(short_entries.value_counts())

# Purely numeric entries
numeric_mask = charges_remaining.str.strip().str.match(r"^\d+$")
numeric_entries = charges_remaining[numeric_mask]
print(f"\n=== Purely numeric entries: {len(numeric_entries)} ===")
if len(numeric_entries) > 0:
    print(numeric_entries.value_counts())

# Check for entries with inline junk (URLs, dates, "PublicLog" text)
# These are valid charges but have artifacts that need stripping
junk_pattern = r"\d+\.\d+\.\d+\.\d+|PublicLog|Page \d+ of \d+|\d{1,2}/\d{1,2}/\d{2,4}"
junk_mask = charges_remaining.str.contains(junk_pattern, case=False, na=False)
junk_entries = charges_remaining[junk_mask]
print(f"\n=== Entries with inline junk (URLs/dates/PublicLog) to strip: {len(junk_entries)} ===")
if len(junk_entries) > 0:
    print(junk_entries.value_counts())

print(f"\n=== Total entries to DELETE (short + numeric): {(short_mask | numeric_mask).sum()} ===")

=== Short entries (< 3 chars): 0 ===

=== Purely numeric entries: 0 ===

=== Entries with inline junk (URLs/dates/PublicLog) to strip: 70 ===
Charges
DEFAULT WARRANT: RECKLESS OPERATION OF MOTOR; VEHICLE c90 S24; DEFAULT WARRANT: MARKED LANES VIOLATION c89 S4A; 172.16.10.215/QED/cadpartner/cad97/newjournal/presslogs.jsp 5/15; 7/3/25, 12:10 AM PublicLog; DEFAULT WARRANT: LICENSE SUSPENDED, OP MV WITH c90 S23                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            1
DEFAULT WARRANT: SHOPLIFTING BY ASPORTATION c266

### Clean up noise
- **Delete** short and purely numeric entries (set to NaN)
- **Strip** inline junk (URLs, IP addresses, dates, "PublicLog Page X of Y") from otherwise valid charges

In [30]:
charges_series = df["Charges"].astype(str)
is_not_null = df["Charges"].notna()

# Delete short and numeric entries
noise_short = is_not_null & (charges_series.str.strip().str.len() < 3)
noise_numeric = is_not_null & charges_series.str.strip().str.match(r"^\d+$")
noise_combined = noise_short | noise_numeric
print(f"Entries deleted (short + numeric): {noise_combined.sum()}")
df.loc[noise_combined, "Charges"] = np.nan

# Strip inline junk from valid charges (URLs, IPs, dates, PublicLog artifacts)
junk_patterns = [
    r"\d+\.\d+\.\d+\.\d+/\S*",        # IP addresses and URL paths
    r"PublicLog\s*Page\s*\d+\s*of\s*\d+",  # "PublicLog Page 3 of 14"
    r"PublicLog",                        # standalone "PublicLog"
    r"QuickSearch\s*Results\s*Page\s*\d+\s*of\s*\d+",  # "QuickSearch Results Page 8 of 14"
    r"\d{1,2}/\d{1,2}/\d{2,4},?\s*\d{1,2}:\d{2}\s*(AM|PM)?",  # dates with times like "5/26/24, 12:25 AM"
    r"\d{1,2}/\d{1,2}/\d{2,4}",        # standalone dates like "10/18/2022"
]

mask_valid = df["Charges"].notna()
for pattern in junk_patterns:
    df.loc[mask_valid, "Charges"] = (
        df.loc[mask_valid, "Charges"]
        .astype(str)
        .str.replace(pattern, "", regex=True, case=False)
    )

# Clean up artifacts left behind (extra semicolons, spaces)
df.loc[mask_valid, "Charges"] = (
    df.loc[mask_valid, "Charges"]
    .str.replace(r";\s*;", ";", regex=True)
    .str.replace(r"^\s*;\s*|\s*;\s*$", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# If stripping junk left an empty string, set to NaN
# (e.g. entries that were ONLY "9/2/2019" or "PublicLog Page 11 of 15")
empty_after_strip = mask_valid & (df["Charges"].astype(str).str.strip() == "")
df.loc[empty_after_strip, "Charges"] = np.nan
print(f"Entries that became empty after junk stripping (set to NaN): {empty_after_strip.sum()}")

print(f"Remaining non-null charges: {df['Charges'].notna().sum()}")

Entries deleted (short + numeric): 0
Entries that became empty after junk stripping (set to NaN): 4
Remaining non-null charges: 814


### Standardize charge text
- Convert to lowercase for case insensitivity
- Strip leading/trailing whitespace
- Remove trailing semicolons
- Collapse multiple spaces into one

In [31]:
mask_valid = df["Charges"].notna()

df.loc[mask_valid, "Charges"] = (
    df.loc[mask_valid, "Charges"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r";\s*$", "", regex=True)   # remove trailing semicolons
    .str.replace(r"\s+", " ", regex=True)     # collapse multiple spaces
    .str.strip()
)

print("Sample standardized charges:")
print(df.loc[mask_valid, "Charges"].head(10))
print(f"\nTotal non-null: {mask_valid.sum()}")

Sample standardized charges:
0                         larceny from person c266 825
1    default warrant: larceny from building c266 s8...
2    default warrant: drug, possess class a c94c $3...
3    trespass c266 s120; warrant charges: default w...
4      defacement malicious wanton property c266 s126a
5                      standard warrant: murder 265 s1
6    default warrant: reckless operation of motor; ...
7    license suspended, op mv with c90 823; marked ...
8    default warrant: unlicensed operation of mv c9...
9    disorderly conduct c272 s53; resisting arrest ...
Name: Charges, dtype: object

Total non-null: 814


### Extract statute references
Statutes appear inline with charge text in patterns like `c90 S24`, `c94C S32A`, `C266 S126A`, or sometimes without the `c` prefix like `265 S15B`.

We extract all statute references into a `statutes` column and create a `cleaned_charges` column with statutes removed.

In [32]:
# Pattern for statutes: optional 'c' + digits + optional letter + space + 's' + digits + optional letters
statute_pattern = r"c?\d+[a-z]?\s+s\d+[a-z]*"

def extract_statutes(charge_str):
    """Extract all statute references from a charge string."""
    if pd.isna(charge_str):
        return np.nan
    matches = re.findall(statute_pattern, str(charge_str), re.IGNORECASE)
    if matches:
        return "; ".join(m.strip() for m in matches)
    return np.nan

def remove_statutes(charge_str):
    """Remove statute references from a charge string and clean up."""
    if pd.isna(charge_str):
        return np.nan
    cleaned = re.sub(statute_pattern, "", str(charge_str), flags=re.IGNORECASE)
    # Clean up leftover artifacts: extra spaces, dangling semicolons
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    cleaned = re.sub(r";\s*;", ";", cleaned)  # collapse double semicolons
    cleaned = re.sub(r"^;\s*|\s*;$", "", cleaned).strip()  # strip leading/trailing semicolons
    return cleaned if cleaned else np.nan

df["statutes"] = df["Charges"].apply(extract_statutes)
df["cleaned_charges"] = df["Charges"].apply(remove_statutes)

print("Sample results:")
sample = df[df["statutes"].notna()][["Charges", "statutes", "cleaned_charges"]].head(10)
print(sample.to_string())

print(f"\nRows with statutes: {df['statutes'].notna().sum()}")
print(f"Rows without statutes: {df['statutes'].isna().sum()}")

Sample results:
                                                                                                                                                                                                                                                                                                                            Charges                          statutes                                                                                                                                                                                                                                                                                        cleaned_charges
1                                                                                                                                                                                                                                                                        default warrant: larceny from building c266 s820; stealing        

### Preview cleaned charges
See what the cleaned charge text looks like and check for any remaining issues.

In [33]:
print("Top 30 most common cleaned charges:")
print(df["cleaned_charges"].value_counts().head(30))

print(f"\nTotal unique cleaned charges: {df['cleaned_charges'].nunique()}")
print(f"\nNull cleaned_charges: {df['cleaned_charges'].isna().sum()}")

Top 30 most common cleaned charges:
cleaned_charges
trespass                                                  18
drug, possess class b                                     14
disorderly conduct                                        14
shoplifting by asportation                                14
license suspended, op mv with                             13
drug, possess class a ; drug, possess class b             10
fugitive from justice on court warrant                     9
drug, possess class a                                      8
oui liquor                                                 6
default warrant: larceny from building ; stealing          6
a&b with dangerous weapon                                  6
motor veh, receive stolen                                  6
default warrant: drug, possess class a                     5
unlicensed operation of mv                                 5
motor veh, malicious damage to                             4
assault w/dangerous weapon       

### Build unique charges reference
Collect all individual charge names across all `charge_N` columns, count their occurrences, and save to `data/charges/unique_charges_reference.csv`.

In [34]:
# Collect individual charge names directly from cleaned_charges
all_charges = (
    df["cleaned_charges"]
    .dropna()
    .astype(str)
    .str.split(";")
    .explode()
    .str.strip()
)

# Remove empty values
all_charges = all_charges[all_charges.ne("")]

# Count each unique charge
charge_counts = (
    all_charges
    .value_counts()
    .rename_axis("charge")
    .reset_index(name="count")
)

print(f"Total unique charges: {len(charge_counts)}")
print("\nTop 30 charges by frequency:")
print(charge_counts.head(30).to_string(index=False))

# Save the reference file
charge_counts.to_csv(UNIQUE_CHARGES_PATH, index=False)

print(f"\nSaved to: {os.path.abspath(UNIQUE_CHARGES_PATH)}")

Total unique charges: 687

Top 30 charges by frequency:
                                         charge  count
                             disorderly conduct     71
                  license suspended, op mv with     55
                     unlicensed operation of mv     50
                          drug, possess class b     49
                                       trespass     46
                               resisting arrest     36
         default warrant: drug, possess class a     36
                          drug, possess class a     36
                      default warrant: trespass     29
                     shoplifting by asportation     26
         fugitive from justice on court warrant     25
                                     oui liquor     24
                     assault w/dangerous weapon     20
         default warrant: drug, possess class b     19
            default warrant: disorderly conduct     19
        standard warrant: drug, possess class a     18
       de

### Save checkpoint 10
Merge the cleaned charge columns back into the full dataframe (including rows that had no charges) and save.

In [35]:
# Merge cleaned columns back into full dataframe
df_out = df_full.copy()

# Create empty columns in df_out for the new fields
new_cols = ["statutes", "cleaned_charges"] + [c for c in df.columns if re.match(r"^(charge|statute)_\d+$", c)]
for col in new_cols:
    df_out[col] = np.nan

# Fill in values from df using the original index alignment
mask = df_full["Charges"].notna() & (df_full["Charges"].astype(str).str.strip() != "")
valid_indices = df_full[mask].index

for col in new_cols:
    if col in df.columns:
        df_out.loc[valid_indices, col] = df[col].values

print("Output shape:", df_out.shape)
print("\nColumns:", df_out.columns.tolist())
print(f"\nNon-null cleaned_charges: {df_out['cleaned_charges'].notna().sum()}")

# Save
df_out.to_csv(OUT_PATH, index=False)
print(f"\nSaved to {OUT_PATH}")

/var/folders/f6/0w5q0_md1413229qftcy9h840000gn/T/ipykernel_61426/2760237554.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan 'c266 s820' 'c94c s834' 'c266 s120; c90 s34j' 'c266 s126a' '265 s1'
 'c90 s24; c89 s4a; c90 s23' 'c89 s4a' 'c90 s6; c89 s9; c90 s10; c90 s10'
 'c272 s53; c268 s32b' 'c265 s13a; 265 s19'
 '265 s15b; 266 s127; c265 s13a' 'c266 s30a' '90 s34j'
 'c265 s15a; c274 s4' '266 s127' 'c94c s832' 'c266 s120'
 'c265 s13a; c266 s120' 'c266 s820; c265 s15a' nan
 'c272 s53; c90 s10; c90 s25; c90 s24' 'c272 s53' nan nan 'c272 s53' nan
 nan 'c266 s1'
 'c94c s34; c94c s34; c94c s34; c266 s30; c266 s60; c94c s34' 'c276 s20a'
 'c94c s34' 'c266 s30a' 'c272 s53' 'c269 s10; 265 s17'
 '272 s53; c268 s32b' '266 s30a' 'c266 s30a' nan '266 s30a' nan
 'c266 s20; c266 s20; c266 s30; c266 s30; c266 s37b; 266 s30'
 'c94c s34; c94c s40' 'c266 s120; c266 s30a; c266 s30' 'c94c s34' nan nan
 'c89 s4a; c90 s2

Output shape: (66741, 22)

Columns: ['Incident #', 'Date', 'Type', 'Location', 'Arrested', 'Location Prefix', 'DOB', 'Charges', 'raw_address', 'cleaned_address', 'latitude', 'longitude', 'geocode_confidence', 'geocoded_at', 'coord_valid', 'person_id', 'category', 'Year', 'crime_severity', 'Age', 'statutes', 'cleaned_charges']

Non-null cleaned_charges: 814

Saved to /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/../0_RecompiledScripts/data/missing_dates_csv/md_checkpoints/checkpoint10_cleaned_charges.csv


### Summary

In [36]:
print("=== Cleaning Summary ===")
print(f"Total rows: {df_out.shape[0]}")
print(f"Rows with charges: {df_out['cleaned_charges'].notna().sum()}")
print(f"Rows without charges: {df_out['cleaned_charges'].isna().sum()}")
print(f"Unique charge types: {charge_counts.shape[0]}")
print(f"\nFiles saved:")
print(f"  - {OUT_PATH}")
print(f"  - {UNIQUE_CHARGES_PATH}")

=== Cleaning Summary ===
Total rows: 66741
Rows with charges: 814
Rows without charges: 65927
Unique charge types: 687

Files saved:
  - /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/../0_RecompiledScripts/data/missing_dates_csv/md_checkpoints/checkpoint10_cleaned_charges.csv
  - /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/../0_RecompiledScripts/data/missing_dates_csv/charges/unique_charges_reference.csv


---
## Iteration 2: Fix remaining junk in charges

After reviewing `unique_charges_reference.csv`, we found ~600+ junk/fragment entries caused by:

1. **Semicolons inside statute references** (e.g. `c272; S53A` instead of `c272 S53A`) — when we split by `;`, these produce orphaned fragments like `S53A`, `with`, `license`, etc.
2. **URLs not caught** — `lawpd-qed.lawpd.cityoflawrence.com/...` domain pattern
3. **Standalone short dates** — `6/14`, `9/17`, etc.
4. **City names** still present — `lawrence, ma`, `lowell, ma`, `methuen, ma`
5. **Address entries** — `address: 155 salem st`
6. **`§` symbol** not matched by statute regex

**Strategy:** Go back to the standardized `Charges` column (before statute extraction), fix the root causes, then re-run extraction and splitting from scratch.

### Step 1: Drop iteration 1 columns and restart from Charges
Drop the `statutes`, `cleaned_charges`, and all `charge_N`/`statute_N` columns from iteration 1. We'll rebuild them after fixing the root causes.

In [37]:
# Drop all iteration 1 derived columns
cols_to_drop = ["statutes", "cleaned_charges"] + [c for c in df.columns if re.match(r"^(charge|statute)_\d+$", c)]
df = df.drop(columns=cols_to_drop, errors="ignore")

print(f"Dropped {len(cols_to_drop)} columns. Remaining columns: {df.columns.tolist()}")
print(f"Non-null Charges: {df['Charges'].notna().sum()}")

Dropped 2 columns. Remaining columns: ['Incident #', 'Date', 'Type', 'Location', 'Arrested', 'Location Prefix', 'DOB', 'Charges', 'raw_address', 'cleaned_address', 'latitude', 'longitude', 'geocode_confidence', 'geocoded_at', 'coord_valid', 'person_id', 'category', 'Year', 'crime_severity', 'Age']
Non-null Charges: 814


### Step 2: Fix semicolons inside statute references
The root cause of most fragments. Entries like `c272; S53A` should be `c272 S53A`. We normalize these BEFORE splitting by semicolon. Also normalize `§` to `S`.

In [38]:
mask_valid = df["Charges"].notna()

before = df["Charges"].copy()

# Fix semicolons inside statute references: "c272; S53A" -> "c272 S53A"
# Also handles "c94C; S32A", "c90; S24", etc.
df.loc[mask_valid, "Charges"] = (
    df.loc[mask_valid, "Charges"]
    .str.replace(r"([c¢]\d+[a-z]?)\s*;\s*([sS§$8]\d+)", r"\1 \2", regex=True, case=False)
)

# Normalize corrupted characters in statute references:
#   ¢ -> c (corrupted chapter prefix)
#   § -> S (section symbol)
# Note: $ -> S and 8 -> S only when they appear as section prefixes
# We handle $ and 8 in the statute regex itself rather than global replace
# since $ and 8 have other valid uses in charge text (dollar amounts, numbers)
df.loc[mask_valid, "Charges"] = (
    df.loc[mask_valid, "Charges"]
    .str.replace("¢", "c", regex=False)
    .str.replace("§", "S", regex=False)
)

# Count how many entries changed
changed = (before.fillna("") != df["Charges"].fillna("")).sum()
print(f"Entries fixed (semicolons in statutes + character normalization): {changed}")

# Show some examples of what changed
diff_mask = (before.fillna("") != df["Charges"].fillna(""))
if diff_mask.any():
    print("\nSample fixes (before -> after):")
    for idx in df[diff_mask].head(5).index:
        print(f"  BEFORE: {before[idx]}")
        print(f"  AFTER:  {df.loc[idx, 'Charges']}")
        print()

Entries fixed (semicolons in statutes + character normalization): 91

Sample fixes (before -> after):
  BEFORE: default warrant: unlicensed operation of mv c90; s10; default warrant: number plate violation c90 s6; default warrant: stop/yield, fail to c89 s9; default warrant: unlicensed operation of mv c90 s10; default warrant: unlicensed operation of mv c90 s10; default warrant: speeding in violation special regulation c90; s18
  AFTER:  default warrant: unlicensed operation of mv c90 s10; default warrant: number plate violation c90 s6; default warrant: stop/yield, fail to c89 s9; default warrant: unlicensed operation of mv c90 s10; default warrant: unlicensed operation of mv c90 s10; default warrant: speeding in violation special regulation c90 s18

  BEFORE: motor veh, malicious damage to ¢266 828
  AFTER:  motor veh, malicious damage to c266 828

  BEFORE: standard warrant: use mv without authority c90; s24; standard warrant: drug, possess class b c94c s34
  AFTER:  standard warrant

### Step 3: Strip additional junk patterns missed in iteration 1
Remove domain URLs, standalone short dates, city/state names, address entries, and other artifacts.

In [39]:
mask_valid = df["Charges"].notna()

# Additional inline junk patterns to strip (these appear WITHIN otherwise valid charge strings)
inline_junk_patterns = [
    r"lawpd-qed\.lawpd\.cityoflawrence\.com\S*",  # domain URLs
    r"\d+\.\d+\.\d+\.\d+/\S*",                    # IP-based URLs (re-apply in case)
    r"QuickSearch\s*Results\s*Page\s*\d+\s*of\s*\d+",
    r"PublicLog\s*Page\s*\d+\s*of\s*\d+",
    r"PublicLog",
    r"\d{1,2}/\d{1,2}/\d{2,4},?\s*\d{1,2}:\d{2}\s*(AM|PM)?",  # dates with times
    r"\d{1,2}/\d{1,2}/\d{2,4}",                    # dates with year
]

for pattern in inline_junk_patterns:
    df.loc[mask_valid, "Charges"] = (
        df.loc[mask_valid, "Charges"]
        .str.replace(pattern, "", regex=True, case=False)
    )

# Clean up artifacts (extra semicolons, spaces)
df.loc[mask_valid, "Charges"] = (
    df.loc[mask_valid, "Charges"]
    .str.replace(r";\s*;", ";", regex=True)
    .str.replace(r"^\s*;\s*|\s*;\s*$", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Now handle entries that are ENTIRELY junk (set to NaN)
# These are full-entry patterns, not inline
fullentry_junk_patterns = [
    r"^\d{1,2}/\d{1,2}$",                          # short dates like "6/14"
    r"^(lawrence|lowell|methuen),?\s*ma$",           # city, state
    r"^lawrence police department$",
    r"^address:\s*.*$",                              # address entries
    r"^transferred in \[dummy.*$",                   # dummy entries
    r"^\d{1,2}/\d{4,},?\s*\d{1,2}:\d{2}\s*(am|pm)?$",  # timestamps as full entry
]

fullentry_junk_mask = pd.Series(False, index=df.index)
for pattern in fullentry_junk_patterns:
    fullentry_junk_mask = fullentry_junk_mask | (
        mask_valid & df["Charges"].str.match(pattern, case=False, na=False)
    )

print(f"Full-entry junk to set to NaN: {fullentry_junk_mask.sum()}")
if fullentry_junk_mask.any():
    print("\nEntries being removed:")
    print(df.loc[fullentry_junk_mask, "Charges"].value_counts().head(20))

df.loc[fullentry_junk_mask, "Charges"] = np.nan

# Also catch entries that became empty after stripping
empty_mask = mask_valid & (df["Charges"].astype(str).str.strip() == "")
df.loc[empty_mask, "Charges"] = np.nan
print(f"Entries empty after stripping: {empty_mask.sum()}")

print(f"\nRemaining non-null charges: {df['Charges'].notna().sum()}")

Full-entry junk to set to NaN: 2

Entries being removed:
Charges
8/17    1
3/12    1
Name: count, dtype: int64
Entries empty after stripping: 0

Remaining non-null charges: 812


### Step 4: Re-run statute extraction
Now that semicolons in statutes are fixed and `§` is normalized, re-extract statutes and create `cleaned_charges`. The regex also includes `§` as an alias for `S`.

In [40]:
# Updated statute pattern — handles corrupted characters from OCR/copy:
#   Chapter prefix: c, C, ¢ (¢ is corrupted c)
#   Section prefix: s, S, §, $, 8 ($ and 8 are corrupted S)
# Pattern: [chapter_prefix]digits[optional_letter] space(s) [section_prefix]digits[optional_letters][optional_parentheticals]
statute_pattern_v2 = r"[c¢]?\d+[a-z]?\s+[sS§$8]\d+[a-z]*(?:\([a-z0-9]+\))*"

def extract_statutes_v2(charge_str):
    """Extract all statute references from a charge string."""
    if pd.isna(charge_str):
        return np.nan
    matches = re.findall(statute_pattern_v2, str(charge_str), re.IGNORECASE)
    if matches:
        return "; ".join(m.strip() for m in matches)
    return np.nan

def remove_statutes_v2(charge_str):
    """Remove statute references from a charge string and clean up."""
    if pd.isna(charge_str):
        return np.nan
    cleaned = re.sub(statute_pattern_v2, "", str(charge_str), flags=re.IGNORECASE)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    cleaned = re.sub(r";\s*;", ";", cleaned)
    cleaned = re.sub(r"^;\s*|\s*;$", "", cleaned).strip()
    return cleaned if cleaned else np.nan

df["statutes"] = df["Charges"].apply(extract_statutes_v2)
df["cleaned_charges"] = df["Charges"].apply(remove_statutes_v2)

print("Sample results:")
sample = df[df["statutes"].notna()][["Charges", "statutes", "cleaned_charges"]].head(10)
print(sample.to_string())

print(f"\nRows with statutes: {df['statutes'].notna().sum()}")
print(f"Rows without statutes: {df['statutes'].isna().sum()}")

# Verify the problematic patterns are now caught
test_patterns = ["¢265", "$17", "834a", "832e", "824", "$32a"]
for pat in test_patterns:
    hits = df["Charges"].dropna().str.contains(pat, na=False).sum()
    hits_cleaned = df["cleaned_charges"].dropna().str.contains(pat, na=False).sum()
    print(f"  '{pat}' in Charges: {hits}, in cleaned_charges: {hits_cleaned}")

Sample results:
                                                                                                                                                                                                                                                                                                                         Charges                                            statutes                                                                                                                                                                                                                                                                       cleaned_charges
0                                                                                                                                                                                                                                                                                                   larceny from person c266 825             

---
## Iteration 3: Clean remaining fragments and junk

After reviewing `unique_charges_reference_v2.csv`, we still have ~100+ problematic entries:

1. **Truncated charge fragments** — partial charge names like `intimate partne`, `damage to`, `malicious c`, `than 10 grams`, `personnel`, `viol`, `wanton c`, `pretense`, etc.
2. **Statute-only fragments** — `c265 sisa`, `$30(1)`, `c269 sioe`, `s14.03`, `c94c s$32e`
3. **Date/time junk** — `7/7124, 1:06 am`, `1/5/8` (corrupted dates)
4. **Generic single words** — `register`, `keep`, `scheme`, `violate`, `provide`, `refuse`, `allow`, `firearm`, `drink`, `use`
5. **System placeholders** — `transferred in [dummy offens`

**Strategy:** Build a comprehensive list of known junk patterns and exact-match fragments, clean them from all `charge_N` columns, then rebuild `cleaned_charges` and save v3.

### Step 1: Define comprehensive fragment patterns and exact matches
Build lists of regex patterns and exact strings that identify junk entries.

In [42]:
# Comprehensive regex patterns for fragments
fragment_patterns_v3 = [
    # Short entries and single chars
    r"^.{1,2}$",                              # 1-2 character entries
    r"^[a-z]$",                               # single letters
    
    # Statute fragments (orphaned sections/chapters)
    r"^[sc]?\d+[a-z]?\s*s[i$]?\d*[a-z]*$",   # statute patterns: c265 sisa, s14.03, etc.
    r"^[sc]\d+[a-z]?$",                       # orphaned chapters: c90, c94c, s14
    r"^\$?\d+\(\d*[a-z]*\)$",                 # $30(1), (1), (b)
    r"^s\d+[a-z]*(\([a-z0-9]+\))?$",          # orphaned sections: s34a, s23
    
    # Numeric and dollar amounts
    r"^[+-]?\$?\d+$",                         # numbers, dollar amounts
    r"^[+-]?\d+\s*grams?$",                   # gram amounts
    r"^\d+\+?\s*grams?$",                     # "10 grams", "28+ grams"
    
    # Dates and times (corrupted or standalone)
    r"^\d{1,2}/\d{1,4}(,?\s*\d{1,2}:\d{2}\s*(am|pm)?)?$",  # dates with optional times
    r"^\d+/\d+/\d+$",                         # standalone dates
    
    # URL/system junk
    r"^lawpd-qed\.",                          # leftover URLs
    r"^address:\s*",                          # address entries
    r"^transferred in \[",                    # system placeholders
    
    # Offense level fragments
    r"^\d+(st|nd|rd|th)\s+off\.?$",           # "2nd off.", "3rd off."
    r"^subsq\.?\s*off\.?$",                   # subsequent offense
    r"^subs\s+off\s*\([a-z]\)$",              # "subs off (f)"
]

# Exact-match fragments (lowercase) — known junk words/phrases
exact_fragments_v3 = {
    # Truncated charge name endings
    "intimate partne", "partne", "intimate pa", "member / intimate partne",
    "household member / intimate pa", "a&b on family / household member /",
    "a&b on family / household", "assault on family / household member / intimate pa",
    "assault on family /", "household member /", "member /",
    
    # Truncated property/damage charges
    "damage to", "malicious c", "wanton c", "wanton property",
    "destruction of prop -$1200,", "destruction of prop +$1200,",
    
    # Truncated drug charges  
    "than 10 grams", "over 14 grams", "class a", "class b", "class c", "class d", "class e",
    "trafficking in", "a, subsq.", "drug, possess to distrib class",
    
    # Truncated weapon charges
    "large cap", "weapon +65", "firearm", "crimes o",
    
    # Truncated warrant fragments
    "conceal id", "personnel", "official, intimidate", "intimdt agg",
    
    # Generic single words (not valid standalone charges)
    "viol", "keep", "register", "scheme", "violate", "provide", "refuse",
    "allow", "drink", "use", "pretense", "operation", "dwelling",
    "improper", "negligent", "stealing", "stealing parts", "asportation",
    "custodian", "cart", "off.", "2nd off.", "3rd off.", "regulation",
    "suffocation", "school/park", "person in fear", "single scheme",
    "under $1200", "for mv", "operate mv with", "oper mv with", "motor veh",
    "with, subsq.off", "+$1200, subsq", "compensation to obtain",
    "grounds,carry", "in mv, drink", "mv, 2nd off", "ist of", "1/5/8",
    "operating, mv, 1st of", "while operating, mv, ist of",
    "obstructed/nontransparent", "displayed", "false/stolen", "subsq. off.",
    "concealing mdse", "race/religion", "order, violate", "mdse, 3rd off.",
    "mdse, 2nd off.", "hypodermic syringe", "distribute", "obtain", "kept",
    
    # Statute-only entries  
    "c265 sisa", "c265 si5b", "c269 sioe", "c94c s$32e", "s14.03", "$30(1)",
    "& s30(1)", "c94c s32e",
    
    # Date/time junk
    "7/7124, 1:06 am", "7/4124, 4:22 am",
    
    # Location/system entries
    "lawrence, ma", "lowell, ma", "methuen, ma",
    "transferred in [dummy offens",
}

print(f"Defined {len(fragment_patterns_v3)} regex patterns")
print(f"Defined {len(exact_fragments_v3)} exact-match fragments")

Defined 17 regex patterns
Defined 105 exact-match fragments


### Step 2: Apply fragment cleanup to all charge columns
Scan all `charge_N` columns and set matching fragments to NaN.

In [43]:
total_cleaned_v3 = 0
cleaned_examples = []


def clean_combined_charge_fragments(value):
    """
    Temporarily split a semicolon-separated charge string,
    remove entries matching the fragment rules,
    and join the remaining charges back together.
    """
    global total_cleaned_v3

    if pd.isna(value) or not str(value).strip():
        return np.nan

    parts = [
        part.strip()
        for part in str(value).split(";")
        if part.strip()
    ]

    kept_parts = []

    for part in parts:
        part_lower = part.lower().strip()

        # Pattern-based fragment detection
        is_fragment = any(
            re.fullmatch(pattern, part, flags=re.IGNORECASE)
            for pattern in fragment_patterns_v3
        )

        # Exact-match fragment detection
        is_fragment = is_fragment or part_lower in exact_fragments_v3

        if is_fragment:
            total_cleaned_v3 += 1
            cleaned_examples.append(part)
        else:
            kept_parts.append(part)

    return "; ".join(kept_parts) if kept_parts else np.nan


df["cleaned_charges"] = df["cleaned_charges"].apply(
    clean_combined_charge_fragments
)

print(
    f"Total fragment entries cleaned in iteration 3: "
    f"{total_cleaned_v3}"
)

print("\nSample entries removed (first 30 unique):")
for entry in sorted(set(cleaned_examples), key=str.lower)[:30]:
    print(f"  - {entry}")

print(
    f"\nNon-null cleaned_charges remaining: "
    f"{df['cleaned_charges'].notna().sum()}"
)

Total fragment entries cleaned in iteration 3: 135

Sample entries removed (first 30 unique):
  - & s30(1)
  - -$1200
  - 1/11
  - 1/12
  - 1/15
  - 1/16
  - 1/17
  - 1/18
  - 10 grams
  - 10/13
  - 10/14
  - 10/15
  - 10/17
  - 10/19
  - 10/21
  - 10/23
  - 11/12
  - 11/14
  - 11/15
  - 11/20
  - 12/13
  - 12/14
  - 13/18
  - 14
  - 14/16
  - 15/17
  - 17/23
  - 2/12
  - 2/13
  - 2/15

Non-null cleaned_charges remaining: 812


### Step 3: Review remaining short entries
Check if any suspicious entries remain after cleanup.

In [44]:
# Temporarily split combined charges for review only
all_remaining_v3 = (
    df["cleaned_charges"]
    .dropna()
    .astype(str)
    .str.split(";")
    .explode()
    .str.strip()
)

# Remove empty strings
all_remaining_v3 = all_remaining_v3[
    all_remaining_v3.ne("")
]

# Review entries shorter than 15 characters
short_remaining_v3 = all_remaining_v3[
    all_remaining_v3.str.len() < 15
]

print("Remaining entries under 15 chars:")
print(short_remaining_v3.value_counts().head(40))

print("\n--- Entries that look like valid charges (keeping these): ---")

valid_short = [
    "trespass",
    "a&b",
    "speeding",
    "assault",
    "oui drugs",
    "stalking",
    "mayhem",
    "conspiracy",
    "murder",
    "kidnapping",
    "carjacking",
]

for charge in valid_short:
    count = (
        all_remaining_v3
        .str.lower()
        .eq(charge.lower())
        .sum()
    )

    if count > 0:
        print(f"  {charge}: {count}")

Remaining entries under 15 chars:
cleaned_charges
trespass          46
oui liquor        24
a&b               11
speeding          10
grams              6
mdse               6
warrant            6
assault            5
mv with            4
mv, 1st of         4
damage             4
oui drugs          3
possess            3
identity fraud     2
property           2
court warrant      2
turn, improper     2
vehicle            1
justice c268       1
kidnapping         1
law enf            1
weapon             1
conspiracy         1
uttering           1
ft of bldg         1
subsq.             1
home invasion      1
gov's warrant      1
with               1
motor vehicle      1
7i17               1
to dist            1
Name: count, dtype: int64

--- Entries that look like valid charges (keeping these): ---
  trespass: 46
  a&b: 11
  speeding: 10
  assault: 5
  oui drugs: 3
  conspiracy: 1
  kidnapping: 1


### Step 4: Save unique charges reference v3 and update checkpoint 13

In [46]:
# GFFI and the df with combined charges 
# Build unique charges reference v3 directly from combined cleaned_charges
all_charges_v3 = (
    df["cleaned_charges"]
    .dropna()
    .astype(str)
    .str.split(";")
    .explode()
    .str.strip()
)

all_charges_v3 = all_charges_v3[all_charges_v3.ne("")]

charge_counts_v3 = (
    all_charges_v3
    .value_counts()
    .rename_axis("charge")
    .reset_index(name="count")
)


# Update checkpoint 13 with v3 cleaned data
df_out = df_full.copy()

new_cols = ["statutes", "cleaned_charges"]

for col in new_cols:
    df_out[col] = np.nan

mask = (
    df_full["Charges"].notna()
    & df_full["Charges"].astype(str).str.strip().ne("")
)

valid_indices = df_full[mask].index

for col in new_cols:
    if col in df.columns:
        df_out.loc[valid_indices, col] = df[col].values

df_out.to_csv(OUT_PATH, index=False)

print("Output shape:", df_out.shape)
print(
    f"Non-null cleaned_charges: "
    f"{df_out['cleaned_charges'].notna().sum()}"
)
print(f"\nOverwritten: {os.path.abspath(OUT_PATH)}")


# Save unique charges reference v3
UNIQUE_CHARGES_V3_PATH = os.path.join(
    CHARGES_DIR,
    "unique_charges_reference_v3.csv"
)

charge_counts_v3.to_csv(UNIQUE_CHARGES_V3_PATH, index=False)

print(f"\nTotal unique charges (v3): {len(charge_counts_v3)}")
print("\nTop 30 charges by frequency:")
print(charge_counts_v3.head(30).to_string(index=False))
print(f"\nSaved to: {os.path.abspath(UNIQUE_CHARGES_V3_PATH)}")

/var/folders/f6/0w5q0_md1413229qftcy9h840000gn/T/ipykernel_61426/1272782020.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['c266 825' 'c266 s820' 'c94c $34; c94c s834'
 'c266 s120; c90 $23; c90 s34j' 'c266 s126a' '265 s1'
 'c90 s24; c89 s4a; c90 s23' 'c90 823; c89 s4a'
 'c90 s10; c90 s6; c89 s9; c90 s10; c90 s10; c90 s18'
 'c272 s53; c268 s32b' 'c265 s13a; 265 s19'
 '265 s15b; 266 s127; c265 s13a' 'c266 s30a' 'c90 823; c90 823; 90 s34j'
 'c265 s15a; c274 s4' '266 s127' 'c94c s832' 'c266 s120'
 'c265 s13a; c266 s120' 'c266 s820; c265 s15a' 'c266 828'
 'c272 s53; c90 s10; c90 s25; c90 s24' 'c272 s53' nan nan 'c272 s53'
 'c94 $32e' 'c94c $34' 'c266 s1'
 'c94c s34; c94c $34; c94c s34; c94c s34; c266 s30; c266 s60; c94c s34'
 'c276 s20a' 'c90 s24; c94c s34' 'c266 s30a' 'c272 s53; c265 $13d'
 'c269 s10; c265 s17' 'c272 s53; c266 $120; c268 s32b' '266 s30a'
 'c266 s30a; c268 834a' nan '266 s30a' 'c90 825

Output shape: (66741, 22)
Non-null cleaned_charges: 812

Overwritten: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/data/missing_dates_csv/md_checkpoints/checkpoint10_cleaned_charges.csv

Total unique charges (v3): 527

Top 30 charges by frequency:
                                         charge  count
                             disorderly conduct     73
                  license suspended, op mv with     57
                     unlicensed operation of mv     50
                          drug, possess class b     49
                                       trespass     46
         default warrant: drug, possess class a     39
                               resisting arrest     36
                          drug, possess class a     36
                      default warrant: trespass     30
    default warrant: shoplifting by asportation     26
                     shoplifting by asportation     26
         fugitive from justice on court warrant   

In [47]:
# Update checkpoint 10 with v3 cleaned data
df_out = df_full.copy()

new_cols = ["statutes", "cleaned_charges"] + [c for c in df.columns if re.match(r"^(charge|statute)_\d+$", c)]
for col in new_cols:
    df_out[col] = np.nan

mask = df_full["Charges"].notna() & (df_full["Charges"].astype(str).str.strip() != "")
valid_indices = df_full[mask].index

for col in new_cols:
    if col in df.columns:
        df_out.loc[valid_indices, col] = df[col].values

df_out.to_csv(OUT_PATH, index=False)

print("Output shape:", df_out.shape)
print(f"Non-null cleaned_charges: {df_out['cleaned_charges'].notna().sum()}")
print(f"\nOverwritten: {OUT_PATH}")

/var/folders/f6/0w5q0_md1413229qftcy9h840000gn/T/ipykernel_61426/2852707948.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['c266 825' 'c266 s820' 'c94c $34; c94c s834'
 'c266 s120; c90 $23; c90 s34j' 'c266 s126a' '265 s1'
 'c90 s24; c89 s4a; c90 s23' 'c90 823; c89 s4a'
 'c90 s10; c90 s6; c89 s9; c90 s10; c90 s10; c90 s18'
 'c272 s53; c268 s32b' 'c265 s13a; 265 s19'
 '265 s15b; 266 s127; c265 s13a' 'c266 s30a' 'c90 823; c90 823; 90 s34j'
 'c265 s15a; c274 s4' '266 s127' 'c94c s832' 'c266 s120'
 'c265 s13a; c266 s120' 'c266 s820; c265 s15a' 'c266 828'
 'c272 s53; c90 s10; c90 s25; c90 s24' 'c272 s53' nan nan 'c272 s53'
 'c94 $32e' 'c94c $34' 'c266 s1'
 'c94c s34; c94c $34; c94c s34; c94c s34; c266 s30; c266 s60; c94c s34'
 'c276 s20a' 'c90 s24; c94c s34' 'c266 s30a' 'c272 s53; c265 $13d'
 'c269 s10; c265 s17' 'c272 s53; c266 $120; c268 s32b' '266 s30a'
 'c266 s30a; c268 834a' nan '266 s30a' 'c90 825

Output shape: (66741, 22)
Non-null cleaned_charges: 812

Overwritten: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/../0_RecompiledScripts/data/missing_dates_csv/md_checkpoints/checkpoint10_cleaned_charges.csv


### Iteration 3 Summary

In [48]:


v3 = charge_counts_v3

print("=== Iteration 3 Summary ===")
print(f"Unique charges v3: {len(v3)}")
print(f"Fragment entries cleaned: {total_cleaned_v3}")
print(f"\nFiles saved:")
print(f"  - {OUT_PATH} (overwritten)")
print(f"  - {UNIQUE_CHARGES_V3_PATH} (new)")

# Show entries removed in v3


=== Iteration 3 Summary ===
Unique charges v3: 527
Fragment entries cleaned: 135

Files saved:
  - /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/../0_RecompiledScripts/data/missing_dates_csv/md_checkpoints/checkpoint10_cleaned_charges.csv (overwritten)
  - /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/0_RecompiledScripts/../0_RecompiledScripts/data/missing_dates_csv/charges/unique_charges_reference_v3.csv (new)
